!pip3 --proxy https://example.invalid/endpoint_0269 install nlopt

In [ ]:
!pip install minio


In [ ]:
from minio import Minio
from minio.error import S3Error
import io
import pandas as pd

In [ ]:
client = Minio(
    'REDACTED_CONFIG_VALUE',  # Change to your MinIO host/IP and port
    access_key='REDACTED_CONFIG_VALUE',         # Replace with your MinIO access key
    secret_key='REDACTED_CONFIG_VALUE',         # Replace with your MinIO secret key
    secure=False                     # Use True if using HTTPS
)

In [ ]:
print(client)

In [ ]:
!pip install mlflow


In [ ]:
!pip install mlflow xgboost plotly scikit-learn pandas seaborn matplotlib numpy joblib scipy pyomo statsmodels minio


In [ ]:
# Import libraries
import mlflow
import mlflow.sklearn
import math
from sklearn.model_selection import GridSearchCV
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import r2_score, mean_squared_error,root_mean_squared_error,mean_absolute_error
from xgboost import XGBRegressor
import xgboost as xgb
import os
from plotly.subplots import make_subplots
from sklearn.ensemble import RandomForestRegressor
import plotly.graph_objects as go
import plotly.io as pio
from collections import Counter
import joblib
from sklearn.preprocessing import StandardScaler
from scipy.optimize import minimize
import re  
from sklearn.preprocessing import MinMaxScaler
import plotly.express as px
import random
from sklearn.feature_selection import SelectFromModel
import warnings
import pyomo.environ as pyo
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

In [ ]:
import os
os.environ['MLFLOW_TRACKING_INSECURE_TLS'] = 'true'
os.environ['MLFLOW_TRACKING_USERNAME'] = 'REDACTED_CONFIG_VALUE'
os.environ['MLFLOW_TRACKING_PASSWORD'] = 'REDACTED_CONFIG_VALUE'
import warnings
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
 
# Configuration
mlflow.set_tracking_uri('https://example.invalid/endpoint_0045')
 
experiment_name = 'Utkal_CGPP_BE_SCC_B1'

experiment = mlflow.get_experiment_by_name(experiment_name)
if experiment is None:
    experiment_id = mlflow.create_experiment(experiment_name)
    print(f"Experiment created with ID: {experiment_id}")
else:
    experiment_id = experiment.experiment_id
    print(f"Experiment found with ID: {experiment_id}")

with mlflow.start_run(experiment_id=experiment_id):
    mlflow.log_param('param1', 5)
    mlflow.log_metric('metric1', 0.85)
print('Run logged successfully.')

In [ ]:
# 2. Define bucket and object name
bucket_name = 'REDACTED_CONFIG_VALUE'
object_name = 'REDACTED_CONFIG_VALUE'         # Replace with actual object name
object_name_1 = 'bronze_data_layer/cgpp/Utkal Boiler Tags Ranges_updated.xlsx'
# 3. Get the object and read as CSV
try:
    # File 1
    response = client.get_object(bucket_name, object_name)
    csv_data = response.read()
    df = pd.read_csv(io.BytesIO(csv_data))  # Convert bytes to DataFrame
    response.close()
    response.release_conn()

    # File 2 - Excel
    response_excel = client.get_object(bucket_name, object_name_1)
    excel_data = response_excel.read()
    range_df = pd.read_excel(io.BytesIO(excel_data))
    
    # Process Excel DataFrame
    range_df = range_df.dropna(subset=['Tag'])
    range_df = range_df.set_index('Tag')
    response_excel.close()
    response_excel.release_conn()
    
    print('Both df and df1 loaded successfully.')
    
except S3Error as err:
    print(f"Error occurred: {err}")

In [ ]:
df.head()

In [ ]:
df_1=df.copy()

In [ ]:
# Parse with format (avoid warning)
df_1['Datetime'] = pd.to_datetime(df_1['Datetime'], dayfirst=True)

# Then format it how you like
df_1['Datetime'] = df_1['Datetime'].dt.strftime('%d-%m-%Y %H:%M')

# If you want to keep it as datetime after formatting
df_1['Datetime'] = pd.to_datetime(df_1['Datetime'], format='%d-%m-%Y %H:%M')

In [ ]:
for col in df_1.select_dtypes(include='object').columns:
    try:
        df_1[col] = pd.to_numeric(df_1[col], errors='coerce')
        print(f"Converted: {col}")
    except Exception as e:
        print(f"Skipped: {col} - {e}")

In [ ]:
# Plot a histogram of a column
df_1['process_signal_0004'].hist(bins=30, edgecolor='black')
plt.title('Histogram')
plt.xlabel('Values')
plt.ylabel('Frequency')
plt.show()

In [ ]:
df_1['process_signal_0004'] = df_1['process_signal_0004'].fillna(
    df_1['process_signal_0004'].median()
)


In [ ]:
columns_to_remove = [
    'process_signal_0014',
    'process_signal_0078',
    'process_signal_0079',
    'process_signal_0077',
    'process_signal_0076',
    'process_signal_0072',
    'process_signal_0073',
    'process_signal_0086',
    'process_signal_0075',
    'process_signal_0074',
    'process_signal_0071'
    'process_signal_0033',
    'process_signal_0034',
    'process_signal_0005',
    'process_signal_0029',
    'process_signal_0019',
    'process_signal_0084',
    'process_signal_0080',
    'process_signal_0082',
    # "process_signal_0012",
    'process_signal_0022'
]

df_1 = df_1.drop(columns=columns_to_remove, errors='ignore')

In [ ]:
excluded_cols = [
    'process_signal_0114',
    'process_signal_0115'
]

# Apply only to columns not in excluded list
df_1.loc[:, ~df_1.columns.isin(excluded_cols)] = df_1.loc[:, ~df_1.columns.isin(excluded_cols)].applymap(
    lambda x: 0 if isinstance(x, (int, float)) and x < 0 else x
)

# Print to check
print(df_1.head())

### convert mg/nm3 to PPM
#### keep SOX and NOX in mg/nm3 and CO in PPM ---> it already set in UOM if required to change so run below code

In [ ]:
# MW_SOX = 64.06   # SO₂ 
# MW_NOX = 46.01   # NO₂
# MW_CO  = 28.01   # CO

# # Conversion factor (at 25°C and 1 atm)
# MOLAR_VOLUME = 24.45  # Liters per mol

# # Conversion formulas
# df_1["process_signal_0039"] = (
#     df_1["process_signal_0039"] * MOLAR_VOLUME / MW_SOX
# )

# df_1["process_signal_0038"] = (
#     df_1["process_signal_0038"] * MOLAR_VOLUME / MW_NOX
# )

# df_1["process_signal_0037"] = (
#     df_1["process_signal_0037"] * MOLAR_VOLUME / MW_CO
# )

In [ ]:
df_1['process_signal_0038'].hist()

In [ ]:
df_1['process_signal_0039'].hist()

In [ ]:
# Step 1: Priority columns (you already defined)
priority_tags = [
    'process_signal_0096',
    'process_signal_0001',
    'process_signal_0028',
    'process_signal_0010',
    'process_signal_0025',
    # "process_signal_0038",
    # "process_signal_0039",
    'process_signal_0040',
    'process_signal_0007.8_HAND_LOSS',
    'process_signal_0006',
    'process_signal_0069',
    'process_signal_0016',
    'process_signal_0017',
    'process_signal_0015',
    'process_signal_0022',
    'process_signal_0081',
    'process_signal_0033',
    'process_signal_0018',
    'process_signal_0032',
    'process_signal_0034',
    'process_signal_0004'
]

# Function to count valid data within range
def count_valid_data(filtered_df, column, range_min, range_max):
    return filtered_df[(filtered_df[column] >= range_min) & (filtered_df[column] <= range_max)].shape[0]

# Start filtering
filtered_data = df_1.copy()

# Step 2: Filter using priority columns first
already_filtered_tags = set()

for tag in priority_tags:
    if tag in filtered_data.columns and tag in range_df.index:
        lower_limit = pd.to_numeric(range_df.loc[tag, 'Lo  Limit'], errors='coerce')
        upper_limit = pd.to_numeric(range_df.loc[tag, 'HI Limit'], errors='coerce')

        if pd.notna(lower_limit) and pd.notna(upper_limit):
            count = count_valid_data(filtered_data, tag, lower_limit, upper_limit)
            print(f"[PRIORITY] Valid count for {tag}: {count}")

            filtered_data = filtered_data[
                (filtered_data[tag] >= lower_limit) & (filtered_data[tag] <= upper_limit)
            ]
            already_filtered_tags.add(tag)

# Step 3: Filter remaining tags in rn not covered in priority list
for tag in range_df.index:
    if tag in filtered_data.columns and tag not in already_filtered_tags:
        lower_limit = pd.to_numeric(range_df.loc[tag, 'Lo  Limit'], errors='coerce')
        upper_limit = pd.to_numeric(range_df.loc[tag, 'HI Limit'], errors='coerce')

        if pd.notna(lower_limit) and pd.notna(upper_limit):
            count = count_valid_data(filtered_data, tag, lower_limit, upper_limit)
            print(f"[OTHER] Valid count for {tag}: {count}")

            filtered_data = filtered_data[
                (filtered_data[tag] >= lower_limit) & (filtered_data[tag] <= upper_limit)
            ]


In [ ]:
filtered_data.columns

In [ ]:
filtered_data.shape

In [ ]:
# Air fuel ratio
filtered_data['process_signal_0012'] = np.where(
    filtered_data['process_signal_0040'] != 0,
    filtered_data['process_signal_0030'] / filtered_data['process_signal_0040'],
    0
)

filtered_data['process_signal_0012'] = (
    filtered_data['process_signal_0012'].replace([np.inf, -np.inf], 0)
)

# secondary and primary flow ratio
filtered_data['process_signal_0023'] = np.where(
    filtered_data['process_signal_0094'] != 0,
    filtered_data['process_signal_0067'] / filtered_data['process_signal_0094'],
    0
)

filtered_data['process_signal_0023'] = (
    filtered_data['process_signal_0023'].replace([np.inf, -np.inf], 0)
)

# specific coal consumption
filtered_data['process_signal_0025'] = (
    filtered_data['process_signal_0040'] /
    filtered_data['process_signal_0096'].replace(0, np.nan)
)

# Replace inf and NaN with 0 — safely
filtered_data['process_signal_0025'] = (
    filtered_data['process_signal_0025']
    .replace([np.inf, -np.inf, np.nan], 0)
)

In [ ]:
filtered_data['process_signal_0023']

In [ ]:
filtered_data.drop(columns=['Datetime'], inplace=True)

In [ ]:
filtered_data['process_signal_0053']

In [ ]:
filtered_data['process_signal_0051']

In [ ]:
filtered_data.shape

In [ ]:
fd = filtered_data.copy() ## use for BE

In [ ]:
fd_1 = filtered_data.copy() ## use for scc

In [ ]:
# columns_to_remove = [
#     "process_signal_0030",
#     "process_signal_0054",
#     "process_signal_0047",
#     "process_signal_0048",
#     "process_signal_0039",
#     "process_signal_0011",
#     "process_signal_0087",
#     "process_signal_0085",
#     "process_signal_0070",
#     "process_signal_0083",
#     "process_signal_0020",
#     "process_signal_0071",
# ]

# fd = fd.drop(columns=columns_to_remove, errors='ignore')

In [ ]:
columns_to_remove = [
    'process_signal_0030',
    'process_signal_0054',
    'process_signal_0047',
    'process_signal_0048',
    'process_signal_0039',
    'process_signal_0011',
    'process_signal_0087',
    'process_signal_0085',
    'process_signal_0070',
    'process_signal_0083',
    'process_signal_0071',
    'process_signal_0089',
    'process_signal_0113',
]

fd = fd.drop(columns=columns_to_remove, errors='ignore')

In [ ]:
filtered_data = filtered_data.dropna()

In [ ]:
fd = fd.dropna()

In [ ]:
fd.shape

### For BE

In [ ]:
be_tag = ['process_signal_0010']

In [ ]:
input_cols = fd.columns.difference(be_tag)
X_be = fd[input_cols]
y_eff = fd[be_tag]

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm 
def calculate_vif(dataframe):
    '\n    Calculate VIF for each feature in the dataframe\n    VIF > 5 indicates moderate multicollinearity\n    VIF > 10 indicates severe multicollinearity\n    '
    vif_data = pd.DataFrame()
    vif_data['Feature'] = dataframe.columns
    vif_data['VIF'] = [variance_inflation_factor(dataframe.values, i) 
                       for i in range(len(dataframe.columns))]
    # Sort by VIF in descending order
    vif_data = vif_data.sort_values(by='VIF', ascending=False)
    return vif_data
# Prepare your data (X should be your feature matrix)
X_with_const = sm.add_constant(X_be)  # Add intercept term for VIF calculation
# Calculate VIF
vif_results = calculate_vif(X_with_const)
print('Variance Inflation Factors:')
print(vif_results)

In [ ]:
s = vif_results.to_csv('VIF_Score_be.csv')

In [ ]:
# Define your target columns
target_cols = [
    'process_signal_0010',
]

# Calculate the correlation matrix
corr = filtered_data.corr()

# Extract correlations for the target columns with other variables
corr_subset = corr[target_cols]

# Save the correlation data to Excel
corr_subset.to_excel('correlation_with_target_variables_1.xlsx', sheet_name='Correlation')

# Plot the heatmap of correlations for the two target variables
plt.figure(figsize=(10, 30))
sns.heatmap(corr_subset, annot=True, cmap='coolwarm', fmt='.2f', square=True, cbar_kws={'shrink': .8})
plt.title('Correlation with Target Variables')
plt.show()

In [ ]:
# Assuming `filtered_data` is already defined
be_tag = ['process_signal_0010']
input_cols = fd.columns.difference(be_tag)

X_be = fd[input_cols]
y_eff = fd[be_tag[0]]  # Convert to Series if it's a DataFrame

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_be, y_eff, test_size=0.2, random_state=42)

# Train XGBoost model
model = xgb.XGBRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Get feature importances
importance_dict = model.get_booster().get_score(importance_type='gain')  # Options: 'weight', 'cover', 'gain', 'total_gain'

# Convert to DataFrame
importance_df = pd.DataFrame({
    'Feature': list(importance_dict.keys()),
    'Importance': list(importance_dict.values())
}).sort_values(by='Importance', ascending=False)

# Display top features
print(importance_df)

# Optional: Plot
plt.figure(figsize=(10, 8))
plt.barh(importance_df['Feature'], importance_df['Importance'])
plt.gca().invert_yaxis()
plt.title('XGBoost Feature Importances (Gain)')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
# List of selected top feature names
Be_selected_features = [
    'process_signal_0032',
    'process_signal_0018',
    'process_signal_0004',
    'process_signal_0069',
    'process_signal_0081',
    'process_signal_0094',
    'process_signal_0025',
    # 'process_signal_0038',
    # 'process_signal_0053', 
    'process_signal_0015',
    'process_signal_0045',
    'process_signal_0016',
    # 'process_signal_0037',
    'process_signal_0102',
    'process_signal_0067',
    'process_signal_0012',
    'process_signal_0023',
    'process_signal_0051',
    'process_signal_0020',
]


In [ ]:
Be_selected_features

In [ ]:
# Target and selected feature subset
X_selected_be = fd[Be_selected_features]
y_eff = fd[be_tag[0]]  # Target variable
X_selected_be=X_selected_be.values
y_eff=y_eff.values

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_selected_be, y_eff, test_size=0.2, random_state=42
)

# Model parameters
params = {
    'n_estimators': 250,
    'learning_rate': 0.01,
    'max_depth': 3,
    'reg_alpha': 0.1,
}
# Start MLflow run with a custom name
with mlflow.start_run(experiment_id=experiment_id, run_name='BE_Model'):
    # Log parameters
    mlflow.log_params(params)

    # Train the model
    model_selected = xgb.XGBRegressor(**params)
    model_selected.fit(X_train, y_train)

    # Predictions
    y_train_pred = model_selected.predict(X_train)
    y_test_pred = model_selected.predict(X_test)

    # Metrics
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    train_r2 = r2_score(y_train, y_train_pred)
    train_mae = mean_absolute_error(y_train, y_train_pred)

    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    test_r2 = r2_score(y_test, y_test_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)

    # Log metrics
    mlflow.log_metrics({
        'train_rmse': train_rmse,
        'train_r2': train_r2,
        'train_mae': train_mae,
        'test_rmse': test_rmse,
        'test_r2': test_r2,
        'test_mae': test_mae
    })

    # Log model
    mlflow.xgboost.log_model(model_selected, artifact_path='model')

    # Print results
    print(f"Training RMSE: {train_rmse:.4f}")
    print(f"Training R²: {train_r2:.4f}")
    print(f"Training MAE: {train_mae:.4f}")
    print(f"Testing RMSE: {test_rmse:.4f}")
    print(f"Testing R²: {test_r2:.4f}")
    print(f"Testing MAE: {test_mae:.4f}")

print('MLflow logging complete.')

In [ ]:
y_test_pred.max()

In [ ]:
# List of selected top feature names
Be_selected_features = [
    'process_signal_0032',
    'process_signal_0018',
    'process_signal_0004',
    'process_signal_0069',
    'process_signal_0081',
    'process_signal_0094',
    'process_signal_0025',
    # 'process_signal_0038',
    # 'process_signal_0053', 
    'process_signal_0015',
    'process_signal_0045',
    'process_signal_0016',
    # 'process_signal_0037',
    'process_signal_0102',
    'process_signal_0067',
    'process_signal_0012',
    'process_signal_0023',
    'process_signal_0051',
    'process_signal_0020',
]

# Extract min and max for each feature
min_max_df = fd[Be_selected_features].agg(['min', 'max']).T
min_max_df.columns = ['Min', 'Max']

# Print the result
print(min_max_df)


In [ ]:
GCV = float(filtered_data['process_signal_0083'].iloc[-1])

In [ ]:
GCV

In [ ]:
# import pyomo.environ as pyo
# import pandas as pd
# import joblib

# # Load trained ML model
# model = joblib.load("model.pkl")
# model_features = list(model.feature_names_in_)
# baseline_values = {feature: 1.0 for feature in model_features}

# # Historical data (assumed loaded externally)
# GCV = 3100  # kcal/kg
# C = 12.5    # example constant for CO loss

# # Pyomo Model
# m = pyo.ConcreteModel()

# # === Variables ===
# m.FlueGasTempAPHOutlet = pyo.Var(bounds=(0, 150))
# m.AirInletTempFDOutlet = pyo.Var(bounds=(15, 60))
# m.MassDryFlueGas = pyo.Var(bounds=(1, 9))
# m.TM = pyo.Var(bounds=(11,12))
# m.H2 = pyo.Var(bounds=(2, 4))
# m.AirHumidityFactor = pyo.Var(bounds=(0.002, 0.3))
# m.ActualAirSupplied = pyo.Var(bounds=(0, 8))
# m.COAPHOutlet = pyo.Var(bounds=(0, 10))
# m.CO2APHOutlet = pyo.Var(bounds=(10,16))
# m.ASH = pyo.Var(bounds=(0, 30))  # Tightened from 50 to 30
# m.ESP_ASH = pyo.Var(bounds=(0, 100))
# m.Bottom_ASH = pyo.Var(bounds=(0, 100))

# # === Constraints (new realistic operational limits) ===
# m.WindBoxDP = pyo.Var(bounds=(10, 120))
# m.FlueGasCO = pyo.Var(bounds=(0, 500))  # ppm
# m.FlueGasNOx = pyo.Var(bounds=(0, 1000))  # ppm
# m.CoalFeedRate = pyo.Var(bounds=(5, 50))  # TPH
# m.MakeupWaterFlow = pyo.Var(bounds=(10, 100))  # m3/hr
# m.DrumPressure = pyo.Var(bounds=(50, 100))  # kg/cm2
# m.ECOGasTemp = pyo.Var(bounds=(100, 250))
# m.ESPOutletGasTemp = pyo.Var(bounds=(100, 250))
# m.PAFanSpeed = pyo.Var(bounds=(2, 50))  # %
# m.MainSteamPressure = pyo.Var(bounds=(120, 300))  # kg/cm2
# m.SuperHeaterTemp = pyo.Var(bounds=(50, 550))  # deg C
# m.WindBoxFanSpeed = pyo.Var(bounds=(10, 70))  # %

# # Sample Constraints tying process variables
# m.con1 = pyo.Constraint(expr=m.FlueGasTempAPHOutlet >= m.AirInletTempFDOutlet + 40)
# m.con2 = pyo.Constraint(expr=m.COAPHOutlet <= 0.1 * (m.COAPHOutlet + m.CO2APHOutlet + 1e-6))  # Partial combustion limited
# m.con3 = pyo.Constraint(expr=m.WindBoxDP >= 300)
# m.con4 = pyo.Constraint(expr=m.FlueGasCO <= 200)
# m.con5 = pyo.Constraint(expr=m.FlueGasNOx <= 250)
# m.con6 = pyo.Constraint(expr=m.CoalFeedRate >= 8)
# m.con7 = pyo.Constraint(expr=m.MakeupWaterFlow <= 80)
# m.con8 = pyo.Constraint(expr=m.DrumPressure >= 90)
# m.con9 = pyo.Constraint(expr=m.ECOGasTemp <= m.FlueGasTempAPHOutlet - 20)
# m.con10 = pyo.Constraint(expr=m.SuperHeaterTemp >= 460)

# # === Loss Equations ===
# m.L1 = pyo.Expression(expr=((m.MassDryFlueGas * 0.24) * (m.FlueGasTempAPHOutlet - m.AirInletTempFDOutlet) / GCV) * 100)
# m.L2 = pyo.Expression(expr=(m.TM * (584 + 0.45 * (m.FlueGasTempAPHOutlet - m.AirInletTempFDOutlet))) / GCV)
# m.L3 = pyo.Expression(expr=8.937 * (584 + 0.45 * (m.FlueGasTempAPHOutlet - m.AirInletTempFDOutlet)) * m.H2 / GCV)
# m.L4 = pyo.Expression(expr=(m.AirHumidityFactor * m.ActualAirSupplied * 0.45 * (m.FlueGasTempAPHOutlet - m.AirInletTempFDOutlet) * 100) / GCV)
# m.L5 = pyo.Expression(expr=(((m.COAPHOutlet * C / 100) / (m.COAPHOutlet + m.CO2APHOutlet + 1e-6)) * 5744 * 100) / GCV)
# m.L6 = pyo.Expression(expr=(0.85 * m.ASH * m.ESP_ASH * 8077) / (100 * GCV))
# m.L7 = pyo.Expression(expr=((m.ASH * 15 / 100) * m.Bottom_ASH * 8077) / (100 * GCV))
# m.L8 = pyo.Expression(expr=((m.ASH * 80 / 10000) * 0.23 * (m.FlueGasTempAPHOutlet - m.AirInletTempFDOutlet) * 100) / GCV)
# m.L9 = pyo.Expression(expr=((m.ASH * 20 / 10000) * 0.23 * (m.FlueGasTempAPHOutlet - m.AirInletTempFDOutlet) * 100) / GCV)

# m.total_loss = pyo.Expression(expr=(
#     m.L1 + m.L2 + m.L3 + m.L4 + m.L5 + m.L6 + m.L7 + m.L8 + m.L9
# ))

# # === Objective ===
# m.obj = pyo.Objective(expr=m.total_loss, sense=pyo.minimize)

# # === Solve ===
# solver = pyo.SolverFactory("ipopt")
# solver.solve(m, tee=True)

# # === Output: Optimized Inputs and Losses ===
# optimized_vars = [
#     "FlueGasTempAPHOutlet", "AirInletTempFDOutlet", "MassDryFlueGas", "TM", "H2",
#     "AirHumidityFactor", "ActualAirSupplied", "COAPHOutlet", "CO2APHOutlet",
#     "ASH", "ESP_ASH", "Bottom_ASH"
# ]
# inputs = {name: pyo.value(getattr(m, name)) for name in optimized_vars}

# print("\n--- Optimized Inputs ---")
# for k, v in inputs.items():
#     print(f"{k}: {v:.2f}")

# print("\n--- Computed Losses ---")
# for i in range(1, 10):
#     print(f"L{i}: {pyo.value(getattr(m, f'L{i}')):.3f}")
# print(f"\n🔥 Total Loss: {pyo.value(m.total_loss):.2f}%")

# # === Predict Efficiency ===
# feature_map = baseline_values.copy()
# loss_feature_mapping = {
#     "process_signal_0069": pyo.value(m.L1),
#     "process_signal_0016": pyo.value(m.L2),
#     "process_signal_0017": pyo.value(m.L3),
#     "process_signal_0015": pyo.value(m.L4),
#     "process_signal_0022": pyo.value(m.L5),
#     "process_signal_0107": pyo.value(m.L6),
#     "process_signal_0106": pyo.value(m.L7),
#     "process_signal_0099": pyo.value(m.L8),
#     "process_signal_0098": pyo.value(m.L9),
#     "process_signal_0032": pyo.value(m.total_loss)
# }
# feature_map.update({k: v for k, v in loss_feature_mapping.items() if k in model_features})
# for feat in model_features:
#     if feat not in feature_map:
#         feature_map[feat] = baseline_values[feat]

# input_df = pd.DataFrame([feature_map])[model_features]
# predicted_efficiency = model.predict(input_df)[0]
# print(f"\n🚀 Predicted Boiler Efficiency: {predicted_efficiency:.2f}%")


#### Scipy based OR

In [ ]:
# Load ML model
logged_model_uri = 'runs:/model_run_0008/model'
model = mlflow.xgboost.load_model(logged_model_uri)

# Model input features
model_features = [
    'process_signal_0032',
    'process_signal_0018',
    'process_signal_0004',
    'process_signal_0069',
    'process_signal_0081',
    'process_signal_0089',
    'process_signal_0025',
    'process_signal_0038',
    'process_signal_0053',
    'process_signal_0015',
    'process_signal_0045',
    'process_signal_0016',
    'process_signal_0037',
    'process_signal_0102',
    'process_signal_0113'
]

# Indexing
var_index = {
    'Tfg': 0, 'Tair': 1, 'Mdfg': 2, 'H2': 3,
    'AAS': 4, 'CO': 5, 'CO2': 6, 'ASH': 7,
    'ESP_ASH': 8, 'TM': 9, 'AF': 10, 'GCV': 11
}
additional_index = {
    'MILL_REJ': 12, 'BLOW_DOWN': 13, 'ESP_RAD': 14,
    'PA_FAN': 15, 'COAL_CONS': 16, 'NOX': 17,
    'ESP_TEMP': 18, 'ECO_TEMP': 19, 'SUPR_HTR': 20, 'WIND_BOX': 21
}

# Bounds
bounds = [
    (120, 150), (15, 60), (1, 9), (2, 4), (0, 8), (0, 10),
    (10, 16), (0, 30), (0, 100), (11, 15), (0.002, 0.3), (2000, 5000),
    (0.033620, 0.062267), (0.225979, 1.171478), (1.123051, 1.750000), 
    (15.070900, 31.268529), (0.194688, 0.372560), (80.205153, 296.965972), 
    (101.658703, 185.228973), (290.003906, 349.981598), 
    (369.804581, 436.672180), (15.203728, 49.999966)
]

# Loss function (all losses)
def calculate_losses(x):
    Tfg,Tair,Mdfg,H2,AAS,CO,CO2,ASH,ESP_ASH,TM,AF,GCV = x[:12]
    L1 = ((Mdfg*0.24)*(Tfg-Tair)/GCV)*100
    L2 = (TM*(584+0.45*(Tfg-Tair)))/GCV
    L3 = 8.937*(584+0.45*(Tfg-Tair))*H2/GCV
    L4 = (AF*AAS*0.45*(Tfg-Tair)*100)/GCV
    L5 = (((CO*GCV/100)/(CO+CO2+1e-6))*5744*100)/GCV
    L6 = (0.85*ASH*ESP_ASH*8077)/(100*GCV)
    L7 = ((ASH*15/100)*10*8077)/(100*GCV)
    L8 = ((ASH*80/10000)*0.23*(Tfg-Tair)*100)/GCV
    L9 = ((ASH*20/10000)*0.23*(Tfg-Tair)*100)/GCV
    total = L1 + L2 + L3 + L4 + L5 + L6 + L7 + L8 + L9
    loss_dict = {
        'Total Loss': total,
        'L1: Dry Flue Gas Loss': L1,
        'L2: Moisture in Fuel': L2,
        'L3: H2 in Fuel': L3,
        'L4: H2O in Air': L4,
        'L5: CO Loss': L5,
        'L6: ESP Ash Loss': L6,
        'L7: Mill Reject Loss': L7,
        'L8: Radiation Loss 1': L8,
        'L9: Radiation Loss 2': L9
    }
    return total, loss_dict

# Optimization objective
def objective(x):
    total_loss, loss_dict = calculate_losses(x)
    feature_map = {
        'process_signal_0032': total_loss,
        'process_signal_0018': x[12],
        'process_signal_0004': x[13],
        'process_signal_0081': x[14],
        'process_signal_0089': x[15],
        'process_signal_0025': x[16],
        'process_signal_0038': x[17],
        'process_signal_0053': x[18],
        'process_signal_0045': x[19],
        'process_signal_0102': x[20],
        'process_signal_0113': x[21],
        'process_signal_0015': loss_dict['L4: H2O in Air'],
        'process_signal_0016': loss_dict['L2: Moisture in Fuel'],
        'process_signal_0069': loss_dict['L1: Dry Flue Gas Loss'],
        'process_signal_0037': x[5]
    }
    input_df = pd.DataFrame([feature_map])[model_features]
    efficiency = model.predict(input_df)[0]
    return -efficiency  # maximize efficiency

# Constraints
cons = [
    {'type': 'ineq', 'fun': lambda x: x[var_index['Tfg']] - x[var_index['Tair']] - 10},
    {'type': 'ineq', 'fun': lambda x: (x[var_index['AAS']] / x[var_index['Mdfg']]) - 1.5},
    {'type': 'ineq', 'fun': lambda x: x[var_index['TM']] - x[var_index['H2']]}
]

# Initial guess
x0 = np.array([120, 30, 5, 3, 4, 1, 12, 10, 50, 12, 0.05, 3000,
               0.04, 0.5, 1.2, 25, 0.3, 150, 120, 310, 390, 30])

# Optimize
res = minimize(objective, x0, method='SLSQP', bounds=bounds, constraints=cons)
if not res.success:
    raise RuntimeError(res.message)

# Result
x_opt = res.x
efficiency = -res.fun
_, loss_dict = calculate_losses(x_opt)

print('\n Optimized Boiler Efficiency:', round(efficiency, 3), '%')

print('\n===  Physical Variables ===')
for var, idx in var_index.items():
    print(f"{var}: {x_opt[idx]:.3f}")

print('\n=== Additional Input Parameters ===')
for name, idx in additional_index.items():
    print(f"{name}: {x_opt[idx]:.3f}")
    
print('\n=== All Individual Losses ===')
for name, val in loss_dict.items():
    print(f"{name}: {val:.3f}")

In [ ]:
import numpy as np
import pandas as pd
import mlflow.xgboost
from scipy.optimize import minimize

# === Load the trained XGBoost model ===
logged_model_uri = 'runs:/model_run_0010/model'
model = mlflow.xgboost.load_model(logged_model_uri)

# === Define model input features ===
model_features = [
    'process_signal_0032',
    'process_signal_0018',
    'process_signal_0004',
    'process_signal_0069',
    'process_signal_0081',
    'process_signal_0094',
    'process_signal_0025',
    'process_signal_0015',
    'process_signal_0045',
    'process_signal_0016',
    'process_signal_0102',
    'process_signal_0067',
    'process_signal_0012',
    'process_signal_0023',
    'process_signal_0051'
]

# === Indexing for optimization variables ===
var_index = {
    'Tfg': 0, 'Tair': 1, 'Mdfg': 2, 'H2': 3,
    'AAS': 4, 'CO': 5, 'CO2': 6, 'ASH': 7,
    'ESP_ASH': 8, 'TM': 9, 'AF': 10, 'GCV': 11
}
additional_index = {
    'MILL_REJ': 12, 'BLOW_DOWN': 13, 'ESP_RAD': 14,
    'PA_FAN': 15, 'COAL_CONS': 16, 'NOX': 17,
    'ESP_TEMP': 18, 'ECO_TEMP': 19, 'SUPR_HTR': 20, 'WIND_BOX': 21,
    'REJECT_QUA': 22, 'REJECT_GCV': 23, 'MAKEUP_WATER': 24,
    'STEAM_GEN': 25, 'STEAM_ENTHALPY': 26, 'FEEDW_ENTHALPY': 27
}

# === Bounds for optimization ===
bounds = [
    (120, 150), (15, 60), (5, 9), (2, 4), (0, 8), (0, 10),
    (10, 16), (0, 30), (0, 100), (11, 15), (0.002, 0.3), (3500, 4000),
    (0.03, 0.06), (0.22, 1.17), (1.12, 1.75),
    (15.07, 31.26), (0.19, 0.37), (80.20, 296.96),
    (101.65, 185.22), (290.00, 349.98),
    (369.80, 436.67), (15.20, 49.99),
    (1000, 1300), (1300, 1500), (10, 100),
    (50, 200), (750, 1000), (100, 200)
]

# === Loss calculation function ===
def calculate_losses(x):
    Tfg, Tair, Mdfg, H2, AAS, CO, CO2, ASH, ESP_ASH, TM, AF, GCV = x[:12]
    RejectQuality = x[22]
    RejectGCV = x[23]
    MakeUpWater = x[24]
    SteamGen = x[25]
    SteamEnthalpy = x[26]
    FeedwEnthalpy = x[27]

    L1 = ((Mdfg * 0.24) * (Tfg - Tair) / GCV) * 100
    L2 = (TM * (584 + 0.45 * (Tfg - Tair))) / GCV
    L3 = 8.937 * (584 + 0.45 * (Tfg - Tair)) * H2 / GCV
    L4 = (AF * AAS * 0.45 * (Tfg - Tair) * 100) / GCV
    L5 = 0.02
    L6 = 1.13
    L7 = ((ASH * 15 / 100) * 10 * 8077) / (100 * GCV)
    L8 = ((ASH * 80 / 10000) * 0.23 * (Tfg - Tair) * 100) / GCV
    L9 = ((ASH * 20 / 10000) * 0.23 * (Tfg - Tair) * 100) / GCV
    L10 = 0.65
    L11 = ((0.8 * MakeUpWater * 300 + 0.2 * MakeUpWater * 150) * 100) / ((SteamGen * (SteamEnthalpy - FeedwEnthalpy)) / 0.85)
    L12 = (RejectQuality * RejectGCV * 100) / ((SteamGen * (SteamEnthalpy - FeedwEnthalpy)) * 1000 / 0.85)

    UL = L6 + L7
    SL = L8 + L9
    total = L1 + L2 + L3 + L4 + L5 + UL + SL + L10 + L11 + L12

    return total, {
        'Total Loss': total, 'L1: Dry Flue Gas Loss': L1, 'L2: Moisture in Fuel': L2,
        'L3: H2 in Fuel': L3, 'L4: H2O in Air': L4, 'L5: Partial Combustion Loss': L5,
        'L6: Unburnt ESP Loss': L6, 'L7: Unburnt bottom ash loss': L7, 'UL: Total Unburnt Loss': UL, 'TL: Total Sensible Loss': SL,
        'L10: Radiation Loss': L10, 'L11: BlowDown Loss': L11, 'L12: Mill Rejection Loss': L12
    }

# === Objective function to maximize efficiency ===
def objective(x):
    total_loss, loss_dict = calculate_losses(x)
    fuel_air_ratio = x[16] / x[21] if x[21] != 0 else 0
    sec_to_pri_flow_ratio = x[21] / x[15] if x[15] != 0 else 0
    esp_temp = x[additional_index['ESP_TEMP']]

    feature_map = {
        'process_signal_0032': total_loss,
        'process_signal_0018': x[12],
        'process_signal_0004': x[13],
        'process_signal_0069': loss_dict['L1: Dry Flue Gas Loss'],
        'process_signal_0081': x[14],
        'process_signal_0094': x[15],
        'process_signal_0025': x[16],
        'process_signal_0015': loss_dict['L4: H2O in Air'],
        'process_signal_0045': x[19],
        'process_signal_0016': loss_dict['L2: Moisture in Fuel'],
        'process_signal_0037': x[5],
        'process_signal_0102': x[20],
        'process_signal_0067': x[21],
        'process_signal_0012': fuel_air_ratio,
        'process_signal_0023': sec_to_pri_flow_ratio,
        'process_signal_0051': esp_temp
    }

    input_df = pd.DataFrame([feature_map])[model_features]
    efficiency = model.predict(input_df)[0]
    return -efficiency  # Maximize efficiency by minimizing negative

# === Constraints for physical feasibility ===
constraints = [
    {'type': 'ineq', 'fun': lambda x: x[var_index['Tfg']] - x[var_index['Tair']] - 10},
    {'type': 'ineq', 'fun': lambda x: (x[var_index['AAS']] / x[var_index['Mdfg']]) - 1.5},
    {'type': 'ineq', 'fun': lambda x: x[var_index['TM']] - x[var_index['H2']]}
]

# === Multiple Optimization Runs ===
n_runs = 20
optimized_solutions = []

for i in range(n_runs):
    x0 = np.array([np.random.uniform(low, high) for (low, high) in bounds])
    res = minimize(objective, x0, method='SLSQP', bounds=bounds, constraints=constraints)
    if res.success:
        optimized_solutions.append(res.x)

if not optimized_solutions:
    raise RuntimeError('All optimization attempts failed.')

# === Compute optimal value ranges ===
optimized_array = np.array(optimized_solutions)
print('\n📈 Optimal Input Variable Ranges (Based on Multiple Runs):')
for i, (low, high) in enumerate(bounds):
    opt_min = optimized_array[:, i].min()
    opt_max = optimized_array[:, i].max()
    print(f"Var {i} - Optimum Min: {opt_min:.3f}, Max: {opt_max:.3f}, Original Bounds: ({low}, {high})")

# === Show best result ===
best_idx = np.argmax([-objective(x) for x in optimized_solutions])
x_opt = optimized_solutions[best_idx]
efficiency = -objective(x_opt)
_, loss_dict = calculate_losses(x_opt)

print(f"\n✅ Best Boiler Efficiency Achieved: {efficiency:.2f}%")
print('\n📊 Optimal Input Variables:')
for name, idx in var_index.items():
    print(f"{name}: {x_opt[idx]:.3f}")
for name, idx in additional_index.items():
    print(f"{name}: {x_opt[idx]:.3f}")

print('\n🔥 Loss Breakdown:')
for name, value in loss_dict.items():
    print(f"{name}: {value:.3f}")


### For SCC 

In [ ]:
# remove all losses in SCC
columns_to_remove = [
    'process_signal_0017',
    'process_signal_0016',
    'process_signal_0033',
    'process_signal_0015',
    'process_signal_0018',
    'process_signal_0069',
    'process_signal_0022',
    'process_signal_0081',
    'process_signal_0034',
    'process_signal_0004',
    'process_signal_0032',
    'process_signal_0086',
    'process_signal_0039',
    'process_signal_0038',
    'process_signal_0037',
    'process_signal_0010',
    'process_signal_0089',
    'process_signal_0113',
    'process_signal_0003'
    
]

fd_1 = fd_1.drop(columns=columns_to_remove, errors='ignore')

### after checking VIF score >10 we remove this parameters from model input

In [ ]:
fd_1.columns

In [ ]:
fd_1 = fd_1.dropna()

In [ ]:
fd_1.isnull().sum()

In [ ]:
scc_tag = ['process_signal_0025']

In [ ]:
input_cols = fd_1.columns.difference(scc_tag)
X_scc = fd_1[input_cols]
y_scc = fd_1[scc_tag]

In [ ]:
X_scc

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
def calculate_vif(dataframe):
    '\n    Calculate VIF for each feature in the dataframe\n    VIF > 5 indicates moderate multicollinearity\n    VIF > 10 indicates severe multicollinearity\n    '
    vif_data = pd.DataFrame()
    vif_data['Feature'] = dataframe.columns
    vif_data['VIF'] = [variance_inflation_factor(dataframe.values, i) 
                       for i in range(len(dataframe.columns))]
    # Sort by VIF in descending order
    vif_data = vif_data.sort_values(by='VIF', ascending=False)
    return vif_data

# Prepare your data (X should be your feature matrix)
X_with_const = sm.add_constant(X_scc)  # Add intercept term for VIF calculation
# Calculate VIF
vif_results = calculate_vif(X_with_const)
print('Variance Inflation Factors:')
print(vif_results)
 

In [ ]:
sc = vif_results.to_csv('VIF_Score_sc.csv')

In [ ]:
# Define your target columns
target_cols = [
    'process_signal_0025',
]

# Calculate the correlation matrix
corr = filtered_data.corr()

# Extract correlations for the target columns with other variables
corr_subset = corr[target_cols]

# Save the correlation data to Excel
corr_subset.to_excel('correlation_with_target_variables_scc.xlsx', sheet_name='Correlation')

# Plot the heatmap of correlations for the two target variables
plt.figure(figsize=(10, 30))
sns.heatmap(corr_subset, annot=True, cmap='coolwarm', fmt='.2f', square=True, cbar_kws={'shrink': .8})
plt.title('Correlation with Target Variables')
plt.show()

In [ ]:
# Assuming `filtered_data` is already defined
scc_tag = ['process_signal_0025']
input_cols = fd_1.columns.difference(scc_tag)

X_scc = fd_1[input_cols]
y_scc = fd_1[scc_tag[0]]  # Convert to Series if it's a DataFrame

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scc, y_scc, test_size=0.2, random_state=42)

# Train XGBoost model
model = xgb.XGBRegressor(n_estimators=100,random_state=42)
model.fit(X_train, y_train)

# Get feature importances
importance_dict = model.get_booster().get_score(importance_type='gain')  # Options: 'weight', 'cover', 'gain', 'total_gain'

# Convert to DataFrame
importance_df = pd.DataFrame({
    'Feature': list(importance_dict.keys()),
    'Importance': list(importance_dict.values())
}).sort_values(by='Importance', ascending=False)

# Display top features
print(importance_df)

# Optional: Plot
plt.figure(figsize=(10, 6))
plt.barh(importance_df['Feature'], importance_df['Importance'])
plt.gca().invert_yaxis()
plt.title('XGBoost Feature Importances (Gain)')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
# List of selected top feature names
scc_selected_features = [
'process_signal_0012',
'process_signal_0030',
'process_signal_0035',
'process_signal_0094',
'process_signal_0067', #Replace it with Wind box only
'process_signal_0023',
# 'process_signal_0039',
# 'process_signal_0113',
# 'process_signal_0037',
# 'process_signal_0038',
# 'process_signal_0089',
# 'process_signal_0010',
# 'process_signal_0053',
'process_signal_0096',
'process_signal_0102',
'process_signal_0045',
'process_signal_0040',
'process_signal_0051', ### similer to APH outlet
'process_signal_0054',
'process_signal_0060',  
'process_signal_0020'
]

In [ ]:
# # Assuming `filtered_data` is already defined
# scc_tag = ["process_signal_0025"]
# input_cols = scc_selected_features

# X_scc = fd_1[input_cols]
# y_scc = fd_1[scc_tag[0]].astype(float)

# # Train-test split
# X_train, X_test, y_train, y_test = train_test_split(X_scc, y_scc, test_size=0.2, random_state=42)

# # Train XGBoost model
# model = xgb.XGBRegressor(n_estimators=100,random_state=42)
# model.fit(X_train, y_train)

# # Get feature importances
# importance_dict = model.get_booster().get_score(importance_type='gain')  # Options: 'weight', 'cover', 'gain', 'total_gain'

# # Convert to DataFrame
# importance_df = pd.DataFrame({
#     'Feature': list(importance_dict.keys()),
#     'Importance': list(importance_dict.values())
# }).sort_values(by='Importance', ascending=False)

# # Display top features
# print(importance_df)

# # Optional: Plot
# plt.figure(figsize=(10, 6))
# plt.barh(importance_df['Feature'], importance_df['Importance'])
# plt.gca().invert_yaxis()
# plt.title('XGBoost Feature Importances (Gain)')
# plt.xlabel('Importance')
# plt.ylabel('Feature')
# plt.tight_layout()
# plt.show()

In [ ]:
# Target and selected feature subset
X_selected_scc = filtered_data[scc_selected_features]
y_scc = filtered_data[scc_tag[0]]  # Assuming this is a Series
X_selected=X_selected_scc.values
y_scc=y_scc.values
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y_scc, test_size=0.2, random_state=42
)

# Model parameters
params = {
    'n_estimators': 100,
    'learning_rate': 0.02,
    'random_state': 42
}

# Start MLflow run with a custom name
with mlflow.start_run(experiment_id=experiment_id, run_name='SCC_Model'):
    # Log parameters
    mlflow.log_params(params)

    # Model training
    model_selected = xgb.XGBRegressor(**params)
    model_selected.fit(X_train, y_train)

    # Predictions
    y_train_pred = model_selected.predict(X_train)
    y_test_pred = model_selected.predict(X_test)

    # Evaluation - Training
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    train_r2 = r2_score(y_train, y_train_pred)
    train_mae = mean_absolute_error(y_train, y_train_pred)

    # Evaluation - Testing
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    test_r2 = r2_score(y_test, y_test_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)

    # Log metrics
    mlflow.log_metrics({
        'train_rmse': train_rmse,
        'train_r2': train_r2,
        'train_mae': train_mae,
        'test_rmse': test_rmse,
        'test_r2': test_r2,
        'test_mae': test_mae
    })

    # Log the model
    mlflow.xgboost.log_model(model_selected, artifact_path='model')

    # Print results
    print(f"Training RMSE: {train_rmse:.4f}")
    print(f"Training R² score: {train_r2:.4f}")
    print(f"Training MAE: {train_mae:.4f}")
    print(f"Testing RMSE: {test_rmse:.4f}")
    print(f"Testing R² score: {test_r2:.4f}")
    print(f"Testing MAE: {test_mae:.4f}")

print('SCC_Model run logged to MLflow.')


In [ ]:
# # Plot and capture the axes
# axes = X_selected.hist(bins=10, figsize=(16, 16), xlabelsize=8, ylabelsize=8)

# # Loop through each axis and set the title font size
# for ax in axes.flatten():
#     ax.title.set_fontsize(8)  # Set the font size for subplot titles

# plt.tight_layout()
# plt.show()

In [ ]:
y_test_pred.min()

In [ ]:
# === Load ML model for SCC===
logged_model_uri = 'runs:/model_run_0012/model'
model = mlflow.xgboost.load_model(logged_model_uri)

# === Feature names ===
model_features = [
    'process_signal_0116',             # x[0]
    'process_signal_0035',           # x[1]
    'process_signal_0039',         # x[2]
    'process_signal_0113',           # x[3]
    'process_signal_0037',          # x[4]
    'process_signal_0038',         # x[5]
    'process_signal_0089',                # x[6]
    'process_signal_0010',            # x[7]
    'process_signal_0053',     # x[8]
    'process_signal_0096',  # x[9]
    'process_signal_0102',       # x[10]
    'process_signal_0045',             # x[11]
    'process_signal_0040',                 # x[12]
    'process_signal_0051',        # x[13]
]

# === Bounds for each feature ===
bounds = [
    (150, 300),   # Total Air Flow
    (30, 100),    # Wind box dp
    (500, 700),   # SOx
    (10, 50),     # Wind box fan avg
    (0, 50),      # CO
    (150, 300),   # NOx
    (18, 30),     # PA fan avg
    (85, 86),     # Final efficiency
    (100, 130),   # ESP outlet gas temp avg
    (150, 300),   # Steam generation
    (380, 420),   # Superheater temp
    (290, 350),   # ECO gas temp
    (40, 90),     # Coal feed
    (240, 300),   # APH outlet temp
]

# === Initial guess ===
x0 = [np.mean([low, high]) for (low, high) in bounds]

# === Objective: maximize predicted efficiency (minimize negative) ===
def objective(x):
    input_df = pd.DataFrame([x], columns=model_features)
    pred = model.predict(input_df)[0]
    return -pred  # maximize efficiency

# === Constraint: air/fuel ratio ≥ 2.11 ===
def air_to_fuel_ratio(x):
    air_flow = x[0]
    coal_flow = x[12]
    return (air_flow / coal_flow) - 2.11

# === Constraint: SCC ≤ max_scc ===
max_scc = 0.202  # Adjust target value as needed
def scc_constraint(x):
    coal_flow = x[12]
    steam_gen = x[9]
    scc = coal_flow / steam_gen
    return max_scc - scc

# === Constraints list ===
constraints = [
    {'type': 'ineq', 'fun': lambda x: x[0] - 150},       # Total Air Flow ≥ 150
    {'type': 'ineq', 'fun': air_to_fuel_ratio},          # Air-to-fuel ratio ≥ 2.11
    {'type': 'ineq', 'fun': lambda x: x[12] - 15.45},    # Coal flow ≥ 15.45
    {'type': 'ineq', 'fun': lambda x: x[9] - 120},       # Steam pressure ≥ 120
    {'type': 'ineq', 'fun': lambda x: x[13] - 190},      # APH outlet ≥ 190
    {'type': 'ineq', 'fun': lambda x: x[1] - 10},        # Wind box dp ≥ 10
    {'type': 'ineq', 'fun': scc_constraint},             # SCC ≤ max_scc
]

# === Run optimization ===
res = minimize(objective, x0, method='SLSQP', bounds=bounds, constraints=constraints)
#print("res-----------",res,res.x,)
# === Check results ===
if not res.success:
    raise RuntimeError(f"Optimization failed: {res.message}")

x_opt = res.x
#print("res-----------",res,res.x,x_opt)
coal_opt = x_opt[12]
steam_opt = x_opt[9]
scc_opt = coal_opt / steam_opt

# === Output ===
print('\n=== Optimization Result ===')
print(f"Minimized SCC: {scc_opt:.4f} (tons/hr steam)")
print(f"Predicted Efficiency: {-res.fun:.4f}")

print('\n=== Optimized Feature Values ===')
for name, val in zip(model_features, x_opt):
    print(f"{name}: {val:.3f}")

In [ ]:
# === Load ML model for SCC===
logged_model_uri = 'runs:/model_run_0014/model'
model = mlflow.xgboost.load_model(logged_model_uri)

model_features = [
    'process_signal_0012',              # x[0]
    'process_signal_0030',                  # x[1]
    'process_signal_0035',                 # x[2]
    'process_signal_0094',            # x[3]
    'process_signal_0067',         # x[4]
    'process_signal_0023',        # x[5]
    'process_signal_0096', # x[6]
    'process_signal_0102',      # x[7]
    'process_signal_0045',            # x[8]
    'process_signal_0040',                 # x[9]
    'process_signal_0051',   # x[10]
    'process_signal_0054',            # x[11]
    'process_signal_0060',              # x[12]
    'process_signal_0020'                          # x[13]
]

bounds = [
    (2.1, 2.8),     # Fuel-air ratio
    (150, 300),     # Total air flow
    (30, 100),      # Wind box dp
    (10, 50),       # PA fan primary air flow
    (10, 50),       # Wind box air flow
    (0.5, 1.5),     # Secondary to primary ratio
    (120, 300),     # Steam flow
    (380, 420),     # Superheater temp
    (290, 350),     # ECO gas temp
    (15.45, 90),    # Coal flow
    (240, 300),     # ESP duct gas temp
    (150, 250),     # FD water flow
    (110, 170),     # FD water temp
    (2, 6),         # Oxygen
]
x0 = [np.mean([low, high]) for (low, high) in bounds]

max_scc = 0.202

def objective(x):
    input_df = pd.DataFrame([x], columns=model_features)
    pred = model.predict(input_df)[0]
    return -pred  # maximize efficiency

def air_to_fuel_ratio(x):
    total_air = x[1]
    coal_flow = x[9]
    return (total_air / coal_flow) - 2.11

def scc_constraint(x):
    coal_flow = x[9]
    steam_gen = x[6]
    scc = coal_flow / steam_gen
    return max_scc - scc

constraints = [
    {'type': 'ineq', 'fun': lambda x: x[1] - 150},      # Total Air Flow ≥ 150
    {'type': 'ineq', 'fun': air_to_fuel_ratio},         # Air-to-fuel ratio ≥ 2.11
    {'type': 'ineq', 'fun': lambda x: x[9] - 15.45},     # Coal Flow ≥ 15.45
    {'type': 'ineq', 'fun': lambda x: x[6] - 120},       # Steam ≥ 120
    {'type': 'ineq', 'fun': lambda x: x[10] - 190},      # APH outlet ≥ 190
    {'type': 'ineq', 'fun': lambda x: x[2] - 10},        # Wind box dp ≥ 10
    {'type': 'ineq', 'fun': scc_constraint},             # SCC ≤ max_scc
]
res = minimize(objective, x0, method='SLSQP', bounds=bounds, constraints=constraints)

if not res.success:
    raise RuntimeError(f"Optimization failed: {res.message}")

x_opt = res.x
coal_opt = x_opt[9]
steam_opt = x_opt[6]
scc_opt = coal_opt / steam_opt

print('\n=== Optimization Result ===')
print(f"Minimized SCC: {scc_opt:.4f} (tons/hr steam)")
print(f"Predicted Efficiency: {-res.fun:.4f}")

print('\n=== Optimized Feature Values ===')
for name, val in zip(model_features, x_opt):
    print(f"{name}: {val:.3f}")

In [ ]:
import numpy as np
from scipy.optimize import minimize
import mlflow.xgboost

# Load the trained ML model
logged_model_uri = 'runs:/model_run_0014/model'
model = mlflow.xgboost.load_model(logged_model_uri)

# Feature names (in correct order)
model_features = [
    'process_signal_0012',              # x[0]
    'process_signal_0030',                  # x[1]
    'process_signal_0035',                 # x[2]
    'process_signal_0094',            # x[3]
    'process_signal_0067',         # x[4]
    'process_signal_0023',        # x[5]
    'process_signal_0096', # x[6]
    'process_signal_0102',      # x[7]
    'process_signal_0045',            # x[8]
    'process_signal_0040',                 # x[9]
    'process_signal_0051',   # x[10]
    'process_signal_0054',            # x[11]
    'process_signal_0060',              # x[12]
    'process_signal_0020'                          # x[13]      
]

# Bounds
bounds = [
    (2.1, 2.8),     # Fuel-air ratio
    (150, 300),     # Total air flow
    (30, 100),      # Wind box dp
    (10, 50),       # PA fan primary air flow
    (10, 50),       # Wind box air flow
    (0.5, 1.5),     # Secondary to primary ratio
    (120, 300),     # Steam flow
    (380, 420),     # Superheater temp
    (290, 350),     # ECO gas temp
    (15.45, 90),    # Coal flow
    (240, 300),     # ESP duct gas temp
    (150, 250),     # FD water flow
    (110, 170),     # FD water temp
    (2, 6),
]

max_scc = 0.202

# Initial guess (midpoints)
x0 = [np.mean(b) for b in bounds]

# Objective function (maximize efficiency)
def objective(x):
    input_df = pd.DataFrame([x], columns=model_features)
    pred = model.predict(input_df)[0]
    return -pred  # maximize efficiency

def air_to_fuel_ratio(x):
    total_air = x[1]
    coal_flow = x[9]
    return (total_air / coal_flow) - 2.11

def scc_constraint(x):
    coal_flow = x[9]
    steam_gen = x[6]
    scc = coal_flow / steam_gen
    return max_scc - scc


constraints = [
    {'type': 'ineq', 'fun': lambda x: x[1] - 150},      # Total Air Flow ≥ 150
    {'type': 'ineq', 'fun': air_to_fuel_ratio},         # Air-to-fuel ratio ≥ 2.11
    {'type': 'ineq', 'fun': lambda x: x[9] - 15.45},     # Coal Flow ≥ 15.45
    {'type': 'ineq', 'fun': lambda x: x[6] - 120},       # Steam ≥ 120
    {'type': 'ineq', 'fun': lambda x: x[10] - 190},      # APH outlet ≥ 190
    {'type': 'ineq', 'fun': lambda x: x[2] - 10},        # Wind box dp ≥ 10
    {'type': 'ineq', 'fun': scc_constraint},             # SCC ≤ max_scc
]

# --- Predict BEFORE optimization ---
initial_eff = model.predict(np.array(x0).reshape(1, -1))[0]

# --- Run Optimization ---
res = minimize(objective, x0, method='SLSQP', bounds=bounds, constraints=constraints)

if not res.success:
    raise RuntimeError(f"Optimization failed: {res.message}")

x_opt = res.x
eff_opt = -res.fun  # flip sign

# === Output Section ===
print('\n=== Boiler Efficiency Optimization ===')
print(f"Initial Predicted Efficiency : {initial_eff:.4f} %")
print(f"Maximized Predicted Efficiency: {eff_opt:.4f} %")

print('\n=== Feature Values Before vs After ===')
for name, before, after in zip(model_features, x0, x_opt):
    print(f"{name:55s} | Before: {before:7.3f} → After: {after:7.3f}")